# Project 1: MTA Turnstile Data

### Background
Week 1 of the Metis Data Science Bootcamp is in the books! Our first project was a group one, in which we were tasked with mining NYC MTA Turnstile data to recommend optimal deployment of a Tech organization's volunteers. The organization wished to maximize mailing list sign-ups, promote their upcoming gala, and solicit potential donors. We were free to use other datasets.

### Team Members

I need to start off by recognizing my team: Brenner Heintz, Hiranya Kumar, and Luke Tibbott. Our collective ideas, feedback, and efforts made a better final product than I would have been able to produce on my own. It was a pleasure to work with them.

### Data:
- [MTA Turnstile Data](http://web.mta.info/developers/turnstile.html)
- 

### MTA Data Cleaning

We started off by downloading one dataset from the MTA website and importing it into python using a csv reader. A scan of the dataframe revealed an issue with the naming of the 'EXITS' column (it contained extra whitespace -- easily removed with `str.strip()`. This may seem like a small detail, but I mention it because **review of imported data is an essential part of data munging**. 

The entry and exit data are given as a series of counter values every 4 hours, *usually* (but not always...see below) at 0:00, 4:00, 8:00, etc. Thus, to get the actual number of entries and exits, we needed to calculate the difference between consecutive rows for a given station's turnstile. 

In [7]:
import pandas as pd
import random
import itertools
import datetime as dt
import matplotlib.pyplot as plt
import calendar
import numpy as np
%matplotlib inline

In [2]:
df1 = pd.read_csv('turnstile_180526.txt')
df2 = pd.read_csv('turnstile_180602.txt')
df3 = pd.read_csv('turnstile_180609.txt')
df4 = pd.read_csv('turnstile_180616.txt')

In [4]:
df = pd.concat([df1, df2, df3, df4])

df.reset_index(inplace=True, drop=True)
df.columns = df.columns.str.strip()

# df.to_csv('raw_turnstile_data.csv')

df['DATETIME'] = pd.to_datetime(df.DATE + ' ' + df.TIME, format='%m/%d/%Y %H:%M:%S')
df['DATE'] = pd.to_datetime(df['DATE'], format='%m/%d/%Y')

df['STATION_KEY'] = df['C/A'] + ' ' + df['UNIT'] + ' ' + df['STATION']

In [5]:
df.sort_values(['STATION_KEY', 'SCP', 'DATETIME'], inplace=True)
df['ENTRY_DIFFS'] = df.groupby(['STATION_KEY','SCP'])['ENTRIES'].diff(periods=-1)*-1
df['EXIT_DIFFS'] = df.groupby(['STATION_KEY','SCP'])['EXITS'].diff(periods=-1)*-1

**Anytime you apply a function** to a dataframe, it's a good idea to **check that it performed as intended**. Reviewing the basic stats of the resulting 'ENTRY_DIFFS' column (or examining a distribution of the values) revealed some very large and some negative values. Examining the large-magnitude differences revealed that they appeared to be due to a resetting of the counter:

In [6]:
df['ENTRY_DIFFS'].describe()

count    7.838270e+05
mean     5.412670e+03
std      5.376402e+06
min     -2.066081e+09
25%      9.000000e+00
50%      7.300000e+01
75%      2.410000e+02
max      2.066520e+09
Name: ENTRY_DIFFS, dtype: float64

In [9]:
indx = df.loc[(df['ENTRY_DIFFS'] > 2E5)][:1].index[0]
df.loc[indx-2:indx+2, ['STATION_KEY', 'DATETIME','ENTRIES','ENTRY_DIFFS']]

,STATION_KEY,DATETIME,ENTRIES,ENTRY_DIFFS
15399,B020 R263 AVENUE H,2018-05-23 04:00:00,92835,9.0
15400,B020 R263 AVENUE H,2018-05-23 08:00:00,92844,18.0
15401,B020 R263 AVENUE H,2018-05-23 12:00:00,92862,523768.0
15402,B020 R263 AVENUE H,2018-05-23 16:00:00,616630,8.0
15403,B020 R263 AVENUE H,2018-05-23 20:00:00,616638,1.0


We chose to filter these outliers out of our dataset. (One of our instructors later mentioned that some turnstile's counters appeared to be counting down instead of up -- so one could simply invert their negative numbers and obtain reasonable data. There are always choices like this to be made in cleaning data.)

In [10]:
# 200,000 entries per 4-hr time period is a reasonable cut-off
clean_df = df[(df['ENTRY_DIFFS'] < 2E5) 
              & (df['ENTRY_DIFFS'] > 0) 
              & (df['EXIT_DIFFS'] < 2E5)
              & (df['EXIT_DIFFS'] > 0)]